# Track 2 — GenAI-Based Classification: Hybrid LLM Pipeline
**American Express Global Business Travel — ML Engineer Take-Home Assignment**

---

## Architecture Overview

```
Incoming article
       │
       ▼
  Preprocessing (lowercase, title+text)
       │
       ▼
  Baseline LR Classifier
       │
       ├─── confidence ≥ threshold ──▶ Final label (source: baseline)  [cheap, fast]
       │
       └─── confidence < threshold ──▶ GPT-4o-mini                     [costly, accurate]
                                            │
                                            ▼
                                   Structured JSON output
                                   (Pydantic schema enforced)
                                            │
                                            ▼
                                       Final label + reasons
```

## Why this design?
- **Cost control**: LLM is only called for uncertain cases (~15–25% of articles).
- **Latency**: 95%+ of articles resolved in < 1 ms via baseline.
- **Accuracy**: LLM handles edge cases the linear model struggles with.
- **Reproducibility**: structured JSON output with strict schema enforcement.

## GenAI Considerations
- **Privacy**: Article text is sent to OpenAI's API. For sensitive documents, use a local model.
- **Cost**: GPT-4o-mini is ~$0.15/1M input tokens. Hybrid routing keeps costs low.
- **Compute**: No GPU required. API-based inference.
- **Reproducibility**: `temperature=0.1` + fixed schema minimises output variance.

## To run without the LLM (no API key needed)
```bash
# Set env var to skip LLM calls — baseline predictions used for all articles
NO_LLM=1 jupyter nbconvert --to notebook --execute track2_llm.ipynb
```
Or set `USE_LLM = False` in the Config cell below.


## 0. Setup & Configuration

In [1]:
import os, sys, re, json, time, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score,
)

from dotenv import load_dotenv
load_dotenv()   # reads OPENAI_API_KEY from .env

# ── Config ──────────────────────────────────────────────────────────────────
SEED           = 42
SAMPLE_N       = 200        # articles sent through hybrid pipeline (cost control)
LLM_THRESHOLD  = 0.80       # route to LLM if baseline confidence < this
LLM_MODEL      = "gpt-4o-mini" #
LLM_MAX_WORDS  = 800        # truncate articles to control token cost

# Set USE_LLM = False to run entirely on baseline (no API key needed)
USE_LLM = os.environ.get("NO_LLM", "0") != "1"

random.seed(SEED)
np.random.seed(SEED)

BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.join(BASE_DIR, "data", "raw")
FIGURES_DIR = os.path.join(BASE_DIR, "outputs", "figures")
METRICS_DIR = os.path.join(BASE_DIR, "outputs", "metrics")
MODELS_DIR  = os.path.join(BASE_DIR, "outputs", "models")

for d in [FIGURES_DIR, METRICS_DIR]:
    os.makedirs(d, exist_ok=True)

def find_csv(name):
    for d in [DATA_DIR, BASE_DIR]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found")

print(f"USE_LLM : {USE_LLM}")
print(f"Model   : {LLM_MODEL}")
print(f"Sample  : {SAMPLE_N} articles")


USE_LLM : True
Model   : gpt-4o-mini
Sample  : 200 articles


###  Model Choice: GPT-4o-mini

For the GenAI baseline, we use **GPT-4o-mini** to balance performance with inference efficiency.

---

###  Why this model

- **Cost-efficient inference**  
  GPT-4o-mini significantly reduces per-request cost compared to full-scale GPT-4 models, making it suitable for large-scale evaluation.

- **Strong enough semantic understanding**  
  Despite being lightweight, it retains sufficient reasoning capability to classify news articles based on tone, context, and narrative cues.

- **Scalability advantage**  
  Enables batch processing of many articles without high computational overhead, which is important for real-world deployment scenarios.

- **Adequate context handling**  
  News articles in this dataset fall within a manageable length range, making them suitable for mini-model context limits.

---

###  Trade-off consideration

While more powerful models (e.g., GPT-4.1) may improve nuanced reasoning, GPT-4o-mini is preferred here due to the balance between:
- cost
- latency
- acceptable classification quality

---

###  Key Insight

This model is used as a **semantic baseline**, not as a production replacement for classical ML models, enabling comparison between:
- TF-IDF + Linear models (efficient production system)
- LLM-based reasoning (semantic understanding benchmark)

In [2]:
# OpenAI client setup (only initialised if USE_LLM=True)
if USE_LLM:
    from openai import OpenAI
    from pydantic import BaseModel, Field
    from typing import Literal, List

    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key or api_key == "your_openai_api_key_here":
        print("WARNING: OPENAI_API_KEY not set. Setting USE_LLM=False.")
        USE_LLM = False
    else:
        client = OpenAI(api_key=api_key)
        print("OpenAI client initialised.")
else:
    print("Running in NO_LLM mode — all predictions from baseline only.")


OpenAI client initialised.


## 1. Data Loading & Preprocessing

*(Same logic as Track 1 — self-contained for reproducibility)*

In [3]:
fake = pd.read_csv(find_csv("Fake.csv"))
true = pd.read_csv(find_csv("True.csv"))
fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true], ignore_index=True)
df["title"]   = df["title"].fillna("").astype(str)
df["text"]    = df["text"].fillna("").astype(str)
df["content"] = (df["title"] + " " + df["text"]).str.lower().str.strip()
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

clean_df = df[["content", "label"]].copy()
print(f"Dataset: {len(clean_df):,} rows | Fake: {(clean_df.label==0).sum()} | Real: {(clean_df.label==1).sum()}")


Dataset: 25,249 rows | Fake: 4057 | Real: 21192


In [4]:
X = clean_df["content"]
y = clean_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")


Train: 20,199  |  Test: 5,050


## 2. Baseline Classifier (Logistic Regression)

Trained to produce **calibrated probabilities** used for confidence-based LLM routing.

In [5]:
MODEL_PATH = os.path.join(MODELS_DIR, "lr_pipeline.joblib")

if os.path.exists(MODEL_PATH):
    baseline_pipeline = joblib.load(MODEL_PATH)
    print("Loaded saved LR pipeline from Track 1.")
else:
    print("Track 1 model not found — retraining baseline inline.")
    baseline_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=50000, ngram_range=(1, 2),
            sublinear_tf=True, min_df=2,
        )),
        ("clf", LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=SEED,
        )),
    ])
    baseline_pipeline.fit(X_train, y_train)
    print("Baseline retrained.")

# Validate on test set
y_pred_baseline = baseline_pipeline.predict(X_test)
print(f"Baseline F1-Macro  : {f1_score(y_test, y_pred_baseline, average='macro'):.4f}")
print(f"Baseline Accuracy  : {accuracy_score(y_test, y_pred_baseline):.4f}")


Track 1 model not found — retraining baseline inline.
Baseline retrained.
Baseline F1-Macro  : 0.9930
Baseline Accuracy  : 0.9962


## 3. LLM Pipeline Design

### Pydantic output schema
Enforces structured JSON from GPT-4o-mini via OpenAI Structured Outputs.


In [6]:
if USE_LLM:
    class ArticleClassification(BaseModel):
        label: Literal["Fake", "Real"] = Field(
            description="Final binary classification."
        )
        confidence: float = Field(
            ge=0.0, le=1.0,
            description="Model confidence between 0 (uncertain) and 1 (certain)."
        )
        reasons: List[str] = Field(
            description="Up to 3 short text-grounded reasons for the decision.",
        )
        needs_human_review: bool = Field(
            description="True if the article is ambiguous, very short, or lacks clear signals."
        )

    print("ArticleClassification schema defined.")
else:
    print("LLM schema skipped (NO_LLM mode).")


ArticleClassification schema defined.


### Domain-aware prompt

The prompt encodes **EDA insights** as classification signals:
- Real news: longer, structured, attributed (Reuters-style vocabulary)
- Fake news: shorter, emotional, vague attribution, social-media references

This grounds the LLM in data-driven signals rather than generic intuition.


In [7]:
SYSTEM_PROMPT = """\
You are an expert NLP classification assistant. Your task is binary text classification.

Task definition:
- Label = Fake: content that is misleading, fabricated, manipulative, or low-credibility
  based solely on its internal textual signals.
- Label = Real: content that is coherent, structured, evidence-based, and consistent
  with credible journalism or factual communication.

IMPORTANT: Judge only from the article text itself. Do not fact-check against external knowledge.

Signals that suggest Fake:
- Sensational, emotionally manipulative, or clickbait phrasing
- Dramatic claims without specific evidence or named attribution
- Vague or anonymous sourcing ("experts say", "sources claim") with no verifiable details
- Internal inconsistency or contradiction
- Social-media artifacts (image captions, retweet-style content, embedded Twitter text)
- Informal vocabulary: "just", "like", "guys", exclamation marks

Signals that suggest Real:
- Named, specific attribution (people, institutions, dates, locations)
- Formal journalistic tone (Reuters-style)
- Balanced, cautious, evidence-based language
- Specific numerical claims tied to named sources
- Consistent and coherent structure

Instructions:
1. Classify the article as exactly one of: Fake or Real.
2. Return a confidence score (0.0 = very uncertain, 1.0 = very certain).
3. Provide up to 3 short reasons grounded in the article text.
4. Set needs_human_review=true if the article is too short, ambiguous, or lacks clear signals.
5. Return ONLY valid JSON matching the required schema.
"""

print("Prompt defined. Length:", len(SYSTEM_PROMPT), "chars")


Prompt defined. Length: 1555 chars


In [8]:
def classify_with_llm(article_text: str) -> dict:
    """Call GPT-4o-mini with structured output schema. Returns a result dict."""
    # Truncate to control token cost
    words = str(article_text).split()[:LLM_MAX_WORDS]
    text  = " ".join(words)

    t0 = time.time()
    response = client.beta.chat.completions.parse(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Article:\n\n{text}"},
        ],
        response_format=ArticleClassification,
        temperature=0.1,
        max_tokens=300,
    )
    latency = time.time() - t0

    parsed  = response.choices[0].message.parsed
    usage   = response.usage

    return {
        "label"               : parsed.label,
        "confidence"          : parsed.confidence,
        "reasons"             : parsed.reasons,
        "needs_human_review"  : parsed.needs_human_review,
        "latency_s"           : round(latency, 3),
        "prompt_tokens"       : usage.prompt_tokens,
        "completion_tokens"   : usage.completion_tokens,
    }

if USE_LLM:
    print("classify_with_llm() defined.")
else:
    print("LLM function skipped (NO_LLM mode).")


classify_with_llm() defined.


## 4. Hybrid Routing Logic

Articles with baseline confidence ≥ `LLM_THRESHOLD` are classified directly.
Low-confidence articles are escalated to GPT-4o-mini.


In [9]:
def hybrid_classify(article_text: str) -> dict:
    """
    Hybrid classifier:
      - High confidence baseline → return immediately (cheap).
      - Low confidence baseline  → call LLM (accurate, slower).
    """
    proba = baseline_pipeline.predict_proba([article_text])[0]
    conf  = float(max(proba))
    bl_label = "Real" if proba.argmax() == 1 else "Fake"

    if not USE_LLM or conf >= LLM_THRESHOLD:
        return {
            "final_label"        : bl_label,
            "source"             : "baseline",
            "baseline_confidence": round(conf, 4),
            "llm_confidence"     : None,
            "reasons"            : [],
            "needs_human_review" : False,
            "latency_s"          : 0.0,
            "prompt_tokens"      : 0,
            "completion_tokens"  : 0,
        }

    llm = classify_with_llm(article_text)
    return {
        "final_label"        : llm["label"],
        "source"             : "llm",
        "baseline_confidence": round(conf, 4),
        "llm_confidence"     : llm["confidence"],
        "reasons"            : llm["reasons"],
        "needs_human_review" : llm["needs_human_review"],
        "latency_s"          : llm["latency_s"],
        "prompt_tokens"      : llm["prompt_tokens"],
        "completion_tokens"  : llm["completion_tokens"],
    }

print(f"Hybrid routing configured (threshold={LLM_THRESHOLD}).")


Hybrid routing configured (threshold=0.8).


## 5. Evaluation on Sample

We evaluate on a **stratified random sample** of the test set.
Using the full test set (~5,700 articles) via LLM would cost ~$0.40 — unnecessary for demonstration.
The sample (200 articles) is stratified to preserve class balance.


In [11]:
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=SAMPLE_N, random_state=SEED)

for _, sample_idx in sss.split(X_test, y_test):
    X_sample = X_test.iloc[sample_idx].reset_index(drop=True)
    y_sample = y_test.iloc[sample_idx].reset_index(drop=True)

print(f"Sample size : {len(X_sample)}")
print(f"Class dist  : {y_sample.value_counts().to_dict()}")

Sample size : 200
Class dist  : {1: 168, 0: 32}


In [12]:
results = []
llm_call_count = 0

for i, (text, true_label) in enumerate(tqdm(zip(X_sample, y_sample), total=len(X_sample))):
    res = hybrid_classify(text)
    if res["source"] == "llm":
        llm_call_count += 1
    results.append({
        "true_label"         : true_label,
        "predicted_label_int": 1 if res["final_label"] == "Real" else 0,
        **res,
    })

results_df = pd.DataFrame(results)
print(f"\nDone. LLM calls made: {llm_call_count} / {len(X_sample)}")
print(f"Baseline-only  : {len(X_sample) - llm_call_count} ({100*(len(X_sample)-llm_call_count)/len(X_sample):.1f}%)")


100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [00:21<00:00,  9.28it/s]


Done. LLM calls made: 8 / 200
Baseline-only  : 192 (96.0%)


In [13]:
y_true_s  = results_df["true_label"].values
y_pred_s  = results_df["predicted_label_int"].values

print("=== Track 2 — Hybrid Pipeline Evaluation ===")
print(f"Accuracy  : {accuracy_score(y_true_s, y_pred_s):.4f}")
print(f"F1-Macro  : {f1_score(y_true_s, y_pred_s, average='macro'):.4f}")
print(f"F1-Wtd    : {f1_score(y_true_s, y_pred_s, average='weighted'):.4f}")
print()
print(classification_report(y_true_s, y_pred_s, target_names=["Fake", "Real"]))


=== Track 2 — Hybrid Pipeline Evaluation ===
Accuracy  : 1.0000
F1-Macro  : 1.0000
F1-Wtd    : 1.0000

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00        32
        Real       1.00      1.00      1.00       168

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [14]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_true_s, y_pred_s)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=ax,
            xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
ax.set_title("Confusion Matrix — Track 2 Hybrid Pipeline", fontweight="bold")
ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "08_track2_confusion_matrix.png"), dpi=150)
plt.close()
print("Saved: 08_track2_confusion_matrix.png")


Saved: 08_track2_confusion_matrix.png


## 6. Cost & Latency Analysis

In [15]:
# GPT-4o-mini pricing (as of Q1 2025)
PRICE_INPUT_PER_M  = 0.150   # $ per 1M input tokens
PRICE_OUTPUT_PER_M = 0.600   # $ per 1M output tokens

llm_rows = results_df[results_df["source"] == "llm"]

if len(llm_rows) > 0:
    total_input   = llm_rows["prompt_tokens"].sum()
    total_output  = llm_rows["completion_tokens"].sum()
    total_cost    = (total_input / 1e6) * PRICE_INPUT_PER_M + (total_output / 1e6) * PRICE_OUTPUT_PER_M
    avg_latency   = llm_rows["latency_s"].mean()

    print(f"LLM calls            : {len(llm_rows)}")
    print(f"Total input tokens   : {total_input:,}")
    print(f"Total output tokens  : {total_output:,}")
    print(f"Total cost (sample)  : ${total_cost:.4f}")
    print(f"Cost / LLM article   : ${total_cost/len(llm_rows):.5f}")
    print(f"Avg LLM latency      : {avg_latency:.2f}s")

    # Extrapolate to full test set
    est_llm_pct  = len(llm_rows) / len(results_df)
    full_test_n  = len(X_test)
    est_llm_full = int(full_test_n * est_llm_pct)
    est_cost_full = est_llm_full * (total_cost / len(llm_rows))
    print(f"\nEstimated LLM calls (full test set) : {est_llm_full}")
    print(f"Estimated cost (full test set)       : ${est_cost_full:.4f}")
else:
    print("No LLM calls made (running in NO_LLM mode or all articles above threshold).")


LLM calls            : 8
Total input tokens   : 8,897
Total output tokens  : 685
Total cost (sample)  : $0.0017
Cost / LLM article   : $0.00022
Avg LLM latency      : 2.61s

Estimated LLM calls (full test set) : 202
Estimated cost (full test set)       : $0.0441


In [16]:
# Latency distribution plot (LLM calls only)
if len(llm_rows) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(llm_rows["latency_s"], bins=20, color="#9b59b6", edgecolor="black")
    axes[0].set_title("LLM Call Latency Distribution", fontweight="bold")
    axes[0].set_xlabel("Latency (seconds)"); axes[0].set_ylabel("Count")

    token_totals = [llm_rows["prompt_tokens"].sum(), llm_rows["completion_tokens"].sum()]
    axes[1].bar(["Input tokens", "Output tokens"], token_totals,
                color=["#3498db", "#e67e22"], edgecolor="black")
    axes[1].set_title("Token Usage (LLM calls)", fontweight="bold")
    axes[1].set_ylabel("Total tokens")
    for bar, val in zip(axes[1].patches, token_totals):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                     f"{val:,}", ha="center", fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "09_cost_latency_analysis.png"), dpi=150)
    plt.close()
    print("Saved: 09_cost_latency_analysis.png")
else:
    print("Latency plot skipped (no LLM calls).")


Saved: 09_cost_latency_analysis.png


## 7. Comparison: Track 1 vs Track 2

In [17]:
# Load Track 1 metrics for comparison
t1_metrics_path = os.path.join(METRICS_DIR, "track1_metrics.csv")

if os.path.exists(t1_metrics_path):
    t1_df = pd.read_csv(t1_metrics_path, index_col=0)
    t1_svc_row = t1_df.loc["LinearSVC (calibrated)"] if "LinearSVC (calibrated)" in t1_df.index else t1_df.iloc[-1]
    t1_f1  = t1_svc_row["F1-Macro"]
    t1_acc = t1_svc_row["Accuracy"]
else:
    # Compute on same sample if Track 1 metrics not saved
    y_pred_t1_sample = baseline_pipeline.predict(X_sample)
    t1_f1  = f1_score(y_sample, y_pred_t1_sample, average="macro")
    t1_acc = accuracy_score(y_sample, y_pred_t1_sample)

t2_f1  = f1_score(y_true_s, y_pred_s, average="macro")
t2_acc = accuracy_score(y_true_s, y_pred_s)

comparison = pd.DataFrame({
    "Track"          : ["Track 1 — LinearSVC (full test)", "Track 2 — Hybrid LLM (sample)"],
    "F1-Macro"       : [round(t1_f1, 4), round(t2_f1, 4)],
    "Accuracy"       : [round(t1_acc, 4), round(t2_acc, 4)],
    "Avg Latency"    : ["< 1 ms", f"{llm_rows['latency_s'].mean():.2f}s (LLM calls)" if len(llm_rows) else "< 1 ms"],
    "Cost/Article"   : ["~$0", f"~${(total_cost/len(llm_rows)):.5f} (LLM cases)" if len(llm_rows) else "~$0"],
    "Deployment"     : ["Single joblib file", "joblib + OpenAI API"],
})
print(comparison.to_string(index=False))
comparison.to_csv(os.path.join(METRICS_DIR, "track1_vs_track2.csv"), index=False)
print("\nSaved: track1_vs_track2.csv")


                          Track  F1-Macro  Accuracy       Avg Latency          Cost/Article          Deployment
Track 1 — LinearSVC (full test)       1.0       1.0            < 1 ms                   ~$0  Single joblib file
  Track 2 — Hybrid LLM (sample)       1.0       1.0 2.61s (LLM calls) ~$0.00022 (LLM cases) joblib + OpenAI API

Saved: track1_vs_track2.csv


### Comparison Discussion

| Dimension | Track 1 (LinearSVC) | Track 2 (Hybrid LLM) |
|---|---|---|
| Accuracy | High (near-perfect on this dataset) | Comparable or better on edge cases |
| Latency | < 1 ms (all articles) | < 1 ms (baseline) / 1–3 s (LLM) |
| Cost | Near zero | Low (LLM only for uncertain articles) |
| Ops complexity | Low — single binary | Medium — requires API key + network |
| Explainability | Low (black-box weights) | High — LLM provides textual reasons |
| Robustness to novel text | Moderate | Higher — LLM reasons from semantics |

**Recommendation**: For high-volume production → **Track 1 (LinearSVC)**.
For analyst-facing workflows where explainability matters → **Track 2 (Hybrid)**.

### GenAI Privacy / Cost / Compute Note
- **Privacy**: Article text leaves your infrastructure when sent to OpenAI. For sensitive content,
  replace `client` with a locally hosted instruct model (e.g. Mistral 7B via Ollama or AWS Bedrock based models).
- **Cost**: At 0.15 dollars/1M input tokens, classifying 1M articles via LLM only costs ~120 dollars at 100% routing.
  With 20% routing (hybrid), this drops to ~$24.
- **Compute**: No GPU required. The OpenAI API handles inference.
- **Reproducibility**: Outputs can vary slightly across runs. Mitigation: `temperature=0.1` + fixed schema.


In [18]:
# Save full results
results_df.to_csv(os.path.join(METRICS_DIR, "track2_predictions.csv"), index=False)
print("Track 2 complete.")
print(f"Figures saved : {FIGURES_DIR}")
print(f"Metrics saved : {METRICS_DIR}")


Track 2 complete.
Figures saved : C:\Users\Swati Gupta\Downloads\MLE_case_study\outputs\figures
Metrics saved : C:\Users\Swati Gupta\Downloads\MLE_case_study\outputs\metrics
